# TimeR4 v8 — Fine-tune LLM suy luận (QLoRA Qwen2.5-7B) theo Equation 11

**Vì sao v8:** v7 chứng minh chuỗi retrieve→rerank đã vững (fact-đúng-ngày vào top-8
tăng 96%→83% lỗi sai-ngày), **nhưng** khi có fact HOÀN HẢO trong top-8, Mistral-7B
chưa-fine-tune vẫn sai **36%**. Nút thắt giờ là **LLM suy luận**.

Paper xác nhận đây là đòn bẩy lớn nhất: trên MultiTQ, **LLaMA2 + TimeR4 KHÔNG
fine-tune = 39.1%**, nhưng **fine-tune LLM (Eq 11) → 72.8%** (gần gấp đôi).

**v8 làm gì:**
1. Đổi LLM nền sang **Qwen2.5-7B-Instruct** (đa ngôn ngữ mạnh hơn Mistral).
2. **QLoRA 4-bit** fine-tune phần *reasoning* trên cặp (facts truy hồi + câu hỏi → đáp án)
   của MultiTQ-VI — đúng tinh thần Equation 11.
3. So sánh **base vs fine-tuned** (cùng retriever, cùng pipeline) để cô lập đúng
   tác động của fine-tune LLM.

> ⚠️ **Kaggle T4 x2 · Internet ON · Persistence=Files only · Restart trước Run All**
> Notebook NẶNG: load 4-bit (~5') + dựng data (~10') + fine-tune (~1.5–3h) +
> chạy test 2 lần (~3h). Tổng ~5–6h. Nếu thiếu giờ: giảm `N_TRAIN`, `TEST_N`.
>
> **Lưu ý khoa học:** để cô lập biến, v8 dùng retriever NỀN đa ngôn ngữ
> (`paraphrase-multilingual-mpnet-base-v2`) cho cả dựng-data lẫn inference.
> Muốn kết quả đỉnh nhất, sau này ghép retriever fine-tune v6 + LLM v8.


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — CÀI THƯ VIỆN + LOAD QWEN2.5-7B (4-bit QLoRA) + LoRA
# ═══════════════════════════════════════════════════════════
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y',
    'peft','sentence-transformers','transformers','huggingface-hub',
    'accelerate','wandb'], capture_output=True)

pkgs = [
    'huggingface_hub==0.23.4', 'peft==0.11.1',
    'transformers==4.44.0', 'sentence-transformers==3.3.1',
    'accelerate==0.34.0', 'faiss-cpu', 'tqdm',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable,'-m','pip','install','-q',pkg], capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"} {pkg}')
    if r.returncode != 0: print('    ', r.stderr[-300:])

# bitsandbytes: nâng lên bản mới để khớp triton 3.x của Kaggle (0.43.1 lỗi 'triton.ops')
rb = subprocess.run([sys.executable,'-m','pip','install','-q','-U','bitsandbytes>=0.44'],
                    capture_output=True, text=True)
print(f'  {"✅" if rb.returncode==0 else "❌"} bitsandbytes (nâng cấp)')
if rb.returncode != 0: print('    ', rb.stderr[-300:])
try:
    import bitsandbytes as _bnb
    print(f'  ✅ bitsandbytes {_bnb.__version__} import OK')
except Exception as e:
    print('  ⚠️ bitsandbytes import lỗi:', str(e)[:200])
    print('     → thử: pip install -U bitsandbytes (Restart Session rồi chạy lại)')

import torch, json, glob, re, random, copy
from contextlib import nullcontext
from tqdm import tqdm
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
print('✅ Import OK')

assert torch.cuda.is_available(), '⚠️ Cần GPU T4! Session options → ACCELERATOR → GPU T4 x2'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
random.seed(42); torch.manual_seed(42)

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
SYS_PROMPT = 'You are a precise temporal question-answering assistant.'

print(f'\nLoading {MODEL_NAME} (4-bit)... (~5-8 phút)')
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map={'': 0},                 # tất cả trên GPU0 cho ổn định khi train
    torch_dtype=torch.float16, attn_implementation='eager')
llm.config.use_cache = False
# Dùng greedy decoding → gỡ tham số sampling để khỏi cảnh báo UserWarning
for _k in ('temperature','top_p','top_k'):
    if hasattr(llm.generation_config, _k):
        setattr(llm.generation_config, _k, None)
import warnings as _warnings
_warnings.filterwarnings('ignore', message='.*do_sample.*')
print(f'✅ Qwen2.5-7B (4-bit) loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ── LoRA ──
llm = prepare_model_for_kbit_training(llm)
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
llm = get_peft_model(llm, lora_cfg)
llm.print_trainable_parameters()
DEVICE = 'cuda'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — CLONE TIMER4 + LOAD MULTITQ-VI (TRAIN + TEST) + TKG
# ═══════════════════════════════════════════════════════════

if not os.path.exists('/kaggle/working/TimeR4'):
    os.system('git clone --depth 1 https://github.com/qianxinying/TimeR4.git /kaggle/working/TimeR4')
os.chdir('/kaggle/working/TimeR4')

triple_list = []
with open('datasets/MultiTQ/kg/full.txt', encoding='utf-8') as f:
    for line in f:
        line = line.strip().replace('_', ' ')
        if not line: continue
        parts = line.split('\t') if '\t' in line else line.split()
        if len(parts) >= 4:
            triple_list.append(parts[:4])
print(f'✅ TKG: {len(triple_list):,} triplets')

found_train = glob.glob('/kaggle/input/**/MultiTQ_vi_train.json', recursive=True)
found_test  = glob.glob('/kaggle/input/**/MultiTQ_vi_test.json', recursive=True)

if not found_test:
    raise FileNotFoundError('MultiTQ_vi_test.json — cần upload')
with open(found_test[0], encoding='utf-8') as f:
    test_questions = json.load(f)
print(f'✅ Test: {found_test[0]} — {len(test_questions):,} câu')

if found_train:
    with open(found_train[0], encoding='utf-8') as f:
        train_questions = json.load(f)
    print(f'✅ Train: {found_train[0]} — {len(train_questions):,} câu')
else:
    train_questions = None
    print('⚠️  Không có train.json')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — TEN + PATCH retrival.py + RERANK v7 + LOAD RETRIEVER
# ═══════════════════════════════════════════════════════════

def normalize_temporal_expression(question):
    text = question.lower()
    spans = []
    for m in re.finditer(r'ngày\s+(\d{1,2})\s+tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'ngày\s+(\d{1,2})/(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(3)}-{int(m.group(2)):02d}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'tháng\s+(\d{1,2})\s+năm\s+(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(2)}-{int(m.group(1)):02d}'))
    for m in re.finditer(r'tháng\s+(\d{1,2})/(\d{4})', text):
        spans.append((m.start(), m.end(), f'{m.group(2)}-{int(m.group(1)):02d}'))
    if not spans: return text
    kept, last_end = [], -1
    for s, e, r in sorted(spans, key=lambda x: -(x[1]-x[0])):
        if s >= last_end:
            kept.append((s, e, r)); last_end = e
    kept.sort(key=lambda x: x[0])
    for s, e, r in reversed(kept):
        text = text[:s] + r + text[e:]
    return text

print('✅ Module TEN loaded')

with open('retrival.py') as f: code = f.read()

patches = [
    ('self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list]',
     'self.triplet_id_list = [[triple[0], triple[1], triple[2], triple[3]] for triple in id_list] if id_list is not None else []'),
    ('self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = embedding_size',
     'self.model = SentenceTransformer(model_name, device="cuda")\n        self.embedding_size = self.model.get_sentence_embedding_dimension()'),
    ('self.full_time = [datetime.strptime(triple[3], "%Y-%m-%d").date() for triple in triple_list]',
     'self.full_time = []\n        for triple in triple_list:\n            t = triple[3] if len(triple) > 3 else "2000-01-01"\n            if len(t) == 4: t += "-01-01"\n            elif len(t) == 7: t += "-01"\n            try: self.full_time.append(datetime.strptime(t[:10], "%Y-%m-%d").date())\n            except: self.full_time.append(datetime.strptime("2000-01-01", "%Y-%m-%d").date())'),
    ('corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]',
     'if corpus_embeddings.ndim == 1:\n            corpus_embeddings = corpus_embeddings[np.newaxis, :]\n        corpus_embeddings = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1)[:, None]'),
]
for old, new in patches:
    if old in code: code = code.replace(old, new)
with open('retrival.py', 'w') as f: f.write(code)
if 'retrival' in sys.modules: del sys.modules['retrival']
from retrival import Retrieval
print('✅ retrival.py patched (4 lỗi kỹ thuật)')


# ── FIX #2: Time-filtering function (Equation 9) ──────────
# ── FIX #2 (v7): Rerank thời gian CHÍNH XÁC — port từ re_rank_single_result ──
from datetime import date as _date, timedelta as _td

def extract_timestamp_from_question(question):
    """Sau TEN: YYYY-MM-DD / YYYY-MM / YYYY. Chỉ chấp nhận tháng 01-12, ngày 01-31
    để KHÔNG bắt nhầm dải năm kiểu '2005-2010' thành '2005-20'."""
    m = re.search(r'(\d{4}(?:-(?:0[1-9]|1[0-2])(?:-(?:0[1-9]|[12]\d|3[01]))?)?)', question)
    return m.group(1) if m else None

# ───────── (GIỮ LẠI hàm CŨ để so sánh trong cell chẩn đoán) ─────────
def time_filter_score_old(tq_str, t_str):
    try:
        def to_date(s):
            s = s[:10]
            if len(s) == 4: s += '-01-01'
            elif len(s) == 7: s += '-01'
            return datetime.strptime(s, '%Y-%m-%d')
        diff = (to_date(tq_str) - to_date(t_str)).days
        return (1 - abs(diff)/3650.0) if diff > 0 else -100
    except Exception:
        return 0

def rerank_facts_old(question, facts, fact_triples_raw=None):
    tq = extract_timestamp_from_question(question)
    if not tq: return facts
    scored = []
    for fact in facts:
        m = re.search(r'in\s+(\d{4}(?:-\d{2}(?:-\d{2})?)?)\.', fact)
        scored.append((fact, time_filter_score_old(tq, m.group(1)) if m else 0))
    valid = [x for x in scored if x[1] != -100]
    pool = valid if valid else scored
    pool.sort(key=lambda x: -x[1])
    return [f for f, s in pool]

# ───────── (MỚI v7) keyword tiếng Việt + chấm điểm theo hướng ─────────
def vi_time_keyword(qnorm):
    ql = qnorm.lower()
    if any(k in ql for k in ['đầu tiên', 'lần đầu', 'sớm nhất']): return 'first'
    if any(k in ql for k in ['cuối cùng', 'gần nhất', 'lần cuối', 'mới nhất']): return 'last'
    if 'trước' in ql: return 'before'
    if 'sau'   in ql: return 'after'
    return 'in'   # vào / ngày / trong / không có → coi như "in"

def q_period(tq):
    """Trả (start, end) bao trùm độ chi tiết mốc thời gian. None nếu không hợp lệ."""
    try:
        tq = tq[:10]
        if len(tq) == 4:
            y = int(tq); return _date(y,1,1), _date(y,12,31)
        if len(tq) == 7:
            y, m = int(tq[:4]), int(tq[5:7])
            if not (1 <= m <= 12): return None
            nxt = _date(y+1,1,1) if m == 12 else _date(y,m+1,1)
            return _date(y,m,1), nxt - _td(days=1)
        if len(tq) == 10:
            return _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10])), _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10]))
    except Exception:
        return None
    return None
    return _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10])), _date(int(tq[:4]),int(tq[5:7]),int(tq[8:10]))

def fact_date(fact):
    try:
        m = re.search(r'in\s+(\d{4})-(\d{2})-(\d{2})', fact)
        if m: return _date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
        m = re.search(r'in\s+(\d{4})-(\d{2})', fact)
        if m: return _date(int(m.group(1)), int(m.group(2)), 1)
        m = re.search(r'in\s+(\d{4})', fact)
        if m: return _date(int(m.group(1)), 1, 1)
    except Exception:
        return None
    return None

def time_score_v7(fd, kw, start, end):
    """1.0 = khớp lý tưởng, -100 = sai hướng (phạt nặng)."""
    if fd is None: return 0.0
    if kw == 'in':
        return 1.0 if (start <= fd <= end) else -100.0
    if kw == 'before':
        if fd < start:
            return 1.0 / (1.0 + (start - fd).days / 30.0)   # càng gần mốc càng cao
        return -100.0
    if kw == 'after':
        if fd > end:
            return 1.0 / (1.0 + (fd - end).days / 30.0)
        return -100.0
    return 0.0   # first/last xử lý riêng

ALPHA = 0.8   # giống paper: final = alpha*sim + (1-alpha)*time

def rerank_facts(question, facts, fact_triples_raw=None):
    """v7: fuse similarity-rank với điểm thời gian theo hướng câu hỏi."""
    tq = extract_timestamp_from_question(question)
    if not tq:
        return facts
    kw = vi_time_keyword(question)
    per = q_period(tq)
    if per is None:
        return facts            # mốc thời gian không phân giải được → giữ nguyên
    start, end = per
    n = max(len(facts), 1)

    # 'first'/'last' → sắp theo thời gian (facts đã liên quan thực thể)
    if kw in ('first', 'last'):
        dated   = [(f, fact_date(f)) for f in facts]
        with_t  = [x for x in dated if x[1] is not None]
        without = [x for x in dated if x[1] is None]
        with_t.sort(key=lambda x: x[1], reverse=(kw == 'last'))
        return [f for f, _ in with_t] + [f for f, _ in without]

    scored = []
    for pos, f in enumerate(facts):
        sim_proxy = 1.0 - pos / n                       # proxy cho similarity (đã xếp sẵn)
        ts = time_score_v7(fact_date(f), kw, start, end)
        scored.append((f, ALPHA * sim_proxy + (1 - ALPHA) * ts))
    scored.sort(key=lambda x: -x[1])
    return [f for f, _ in scored]

print('✅ Rerank v7 (chính xác theo hướng trước/sau/vào + granularity) sẵn sàng')

def get_q(x): return x.get('question', x.get('Question', ''))

temporal_q = [q for q in test_questions if normalize_temporal_expression(get_q(q)) != get_q(q).lower()]
questions_1000 = temporal_q[:1000]
print(f'\nDùng {len(questions_1000)} câu test (cùng tập với các lần chạy trước)')

# ── Load retriever NỀN đa ngôn ngữ (dùng cho cả dựng-data lẫn inference) ──
RETRIEVER_BASE = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
# Nếu đã có retriever fine-tune v6 trong input/working thì ưu tiên dùng (tốt hơn):
import glob as _glob
_cand = _glob.glob('/kaggle/input/**/finetuned_retriever_v6', recursive=True) + \
        (['/kaggle/working/models/finetuned_retriever_v6'] if os.path.exists('/kaggle/working/models/finetuned_retriever_v6') else [])
RETRIEVER_PATH = _cand[0] if _cand else RETRIEVER_BASE
print(f'✅ Retriever dùng: {RETRIEVER_PATH}')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — PROMPT (Figure 5/6) + GEN theo CHAT TEMPLATE QWEN
# rewrite dùng BASE (tắt adapter) ; reasoning dùng adapter đã fine-tune
# ═══════════════════════════════════════════════════════════
def build_rewrite_prompt(fact, question):
    return ('Replace the temporal fact in questions with explicit timestamps '
        'from the provided facts or your knowledge without any explanation. '
        'If you are not sure about the answer, return the original questions.\n\n'
        'For instance, from the fact:\n'
        '"[Juan Carlos I, Praise or endorse, Vietnam, 2006-02-22]",\n'
        'We can modify the question:\n'
        '"After Vietnam, who was the first to praise Juan Carlos I?"\n'
        'to "After 2006-02-22, who was the first to praise Juan Carlos I?"\n\n'
        'Here is your turn:\n'
        f'Facts: {fact}\nQuestion: {question}')

def build_reasoning_prompt(facts, question):
    return ('Based on the historical facts, please answer the given question. '
        'Please keep the answer as simple as possible and return all the '
        'possible answers as a list.\n'
        f'Historical facts: {facts}\nQuestion: {question}')

def _qwen_gen(user_content, max_new_tokens=64, use_adapter=True, max_in=1500):
    msgs = [{'role':'system','content':SYS_PROMPT},
            {'role':'user','content':user_content}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inp = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_in).to(DEVICE)
    ctx = (llm.disable_adapter() if (not use_adapter and hasattr(llm,'disable_adapter'))
           else nullcontext())
    with torch.no_grad(), ctx:
        out = llm.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# rewrite: hành vi gốc (tắt adapter); reasoning: adapter fine-tune
USE_FT = True   # cell run sẽ bật/tắt để so base vs fine-tuned
def call_llm(prompt, max_new_tokens=60):
    return _qwen_gen(prompt, max_new_tokens=max_new_tokens, use_adapter=False).split('\n')[0].strip()
def gen_answer(prompt, max_new_tokens=64):
    raw = _qwen_gen(prompt, max_new_tokens=max_new_tokens, use_adapter=USE_FT)
    return raw.split('\n')[0].strip()

def parse_list_answer(raw):
    m = re.findall(r"['\"]([^'\"]+)['\"]", raw)
    return m if m else [raw.strip('[]\'\" .')]

def get_gold(x): return x.get('answer', x.get('answers', x.get('Answer', x.get('answers_simple','?'))))

def gold_objects(item):
    g = get_gold(item)
    if isinstance(g, str):
        inner = re.findall(r"['\"]([^'\"]+)['\"]", g); return inner if inner else [g]
    return g if isinstance(g, list) else [g]

import string as _string
def _normalize_ans(s):
    """Khớp ĐÚNG metric của tác giả (predict_answer.py normalize):
    lowercase + bỏ dấu câu + bỏ mạo từ a/an/the + bỏ <pad> + gộp khoảng trắng.
    Bỏ dấu câu giúp gold ICEWS 'Foreign Affairs (South Korea)' -> 'foreign affairs south korea'."""
    s = str(s).lower()
    s = ''.join(ch for ch in s if ch not in set(_string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = re.sub(r'\b(pad)\b', ' ', s)
    return ' '.join(s.split())

def evaluate_hit(pred_raw, gold):
    """Theo tác giả: chuẩn hoá rồi kiểm tra gold (chuẩn hoá) NẰM TRONG prediction."""
    pred_str = _normalize_ans(pred_raw)
    golds = gold if isinstance(gold, list) else [gold]
    for g in golds:
        ng = _normalize_ans(g)
        if ng and ng in pred_str:
            return True
    return False

print('✅ Prompt + Qwen chat-template gen sẵn sàng')
print('--- smoke test reasoning ---')
print(gen_answer(build_reasoning_prompt("['UK hasLeader David Cameron in 2012-05-01.']", 'Who led UK in 2012?')))


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — DỰNG DỮ LIỆU FINE-TUNE REASONING (Equation 11)
# Cho mỗi câu train: TEN → retrieve(top-50) → rerank → top-8 facts
#   prompt = Reasoning prompt (Figure 6) ; target = gold answer dạng list
# ═══════════════════════════════════════════════════════════
assert train_questions is not None, 'Cần MultiTQ_vi_train.json'

N_TRAIN   = 4000     # số câu train dùng để fine-tune (giảm nếu thiếu giờ)
N_RETR    = 50
TOPK      = 8

def gold_list_str(item):
    g = get_gold(item)
    if isinstance(g, str):
        inner = re.findall(r"['\"]([^'\"]+)['\"]", g)
        g = inner if inner else [g]
    if not isinstance(g, list): g = [g]
    return '[' + ', '.join(f"'{str(x)}'" for x in g) + ']'

# lọc câu có đáp án rõ ràng + có biểu thức thời gian (giống phân phối test)
train_pool = [q for q in train_questions
              if get_gold(q) not in (None, '?', '', [])][:N_TRAIN*2]
random.shuffle(train_pool)
train_pool = train_pool[:N_TRAIN]
print(f'Dựng training data từ {len(train_pool)} câu train...')

q_tr  = [normalize_temporal_expression(get_q(q)) for q in train_pool]
r_tr  = Retrieval(RETRIEVER_PATH, q_tr, triple_list, None, None)
d_tr, c_tr = r_tr.compute_similarity(n=N_RETR)
res_tr = r_tr.get_result(d_tr, c_tr, q_tr, re_rank=False)

train_examples = []
for i, item in enumerate(tqdm(train_pool, desc='Build pairs')):
    facts = res_tr[i].get('fact') or []
    facts = rerank_facts(q_tr[i], facts)[:TOPK]
    if not facts:
        continue
    prompt = build_reasoning_prompt(facts, q_tr[i])
    target = gold_list_str(item)
    train_examples.append({'prompt': prompt, 'target': target})

print(f'✅ {len(train_examples)} cặp training.')
print('\n--- VÍ DỤ ---')
print('PROMPT:', train_examples[0]['prompt'][:240], '...')
print('TARGET:', train_examples[0]['target'])

del r_tr; torch.cuda.empty_cache()


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — QLoRA FINE-TUNE (chỉ tính loss trên phần đáp án)
# ═══════════════════════════════════════════════════════════
MAX_LEN = 1024
EPOCHS  = 1

class SFTDataset(torch.utils.data.Dataset):
    def __init__(self, examples):
        self.rows = []
        for ex in examples:
            msgs = [{'role':'system','content':SYS_PROMPT},
                    {'role':'user','content':ex['prompt']}]
            prompt_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True)
            target_ids = tokenizer(ex['target'] + tokenizer.eos_token, add_special_tokens=False)['input_ids']
            input_ids = (prompt_ids + target_ids)[:MAX_LEN]
            labels    = ([-100]*len(prompt_ids) + target_ids)[:MAX_LEN]
            if len(input_ids) <= len(prompt_ids):   # đáp án bị cắt hết → bỏ
                continue
            self.rows.append({'input_ids':input_ids,'labels':labels})
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    maxlen = max(len(x['input_ids']) for x in batch)
    pad = tokenizer.pad_token_id
    ids, lab, am = [], [], []
    for x in batch:
        d = maxlen - len(x['input_ids'])
        ids.append(x['input_ids'] + [pad]*d)
        lab.append(x['labels'] + [-100]*d)
        am.append([1]*len(x['input_ids']) + [0]*d)
    return {'input_ids':torch.tensor(ids), 'labels':torch.tensor(lab),
            'attention_mask':torch.tensor(am)}

ds = SFTDataset(train_examples)
print(f'Dataset: {len(ds)} mẫu (sau khi lọc cắt cụt)')

args = TrainingArguments(
    output_dir='/kaggle/working/qwen_lora_ckpt',
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_ratio=0.03,
    logging_steps=20, save_strategy='no', fp16=True,
    gradient_checkpointing=True, optim='paged_adamw_8bit',
    report_to='none', lr_scheduler_type='cosine')

trainer = Trainer(model=llm, args=args, train_dataset=ds, data_collator=collate)
llm.config.use_cache = False
print('🚀 Bắt đầu fine-tune (~1.5–3h)...')
trainer.train()

LORA_DIR = '/kaggle/working/qwen_lora_reasoning_v8'
llm.save_pretrained(LORA_DIR)
print(f'✅ Đã lưu LoRA adapter: {LORA_DIR}')
llm.config.use_cache = True


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — PIPELINE v8 (giống v7, LLM = Qwen fine-tuned)
# ═══════════════════════════════════════════════════════════
def run_pipeline_v8(qs_raw, triples, retriever_path, n_retrieve=50, top_k_facts=8, label=''):
    qs = copy.deepcopy(qs_raw)
    print(f'\n{"="*60}\n  {label}\n{"="*60}')
    q_list = [get_q(q) for q in qs]
    q_norm = [normalize_temporal_expression(q) for q in q_list]

    print('[1/4] Retrieve (FKS)...')
    r1 = Retrieval(retriever_path, q_norm, triples, None, None)
    d1, c1 = r1.compute_similarity(n=n_retrieve)
    bg = r1.get_result(d1, c1, q_norm, re_rank=False)

    print('[2/4] Rewrite (base, tắt adapter)...')
    for i in tqdm(range(len(qs)), desc='Rewrite', leave=False):
        q = get_q(qs[i]); f = (bg[i].get('fact') or [''])[0]
        resp = call_llm(build_rewrite_prompt(f, q))
        qs[i]['question'] = resp if resp else q

    print('[3/4] TA-Retrieve + Rerank...')
    q2 = [normalize_temporal_expression(get_q(q)) for q in qs]
    r2 = Retrieval(retriever_path, q2, triples, None, None)
    d2, c2 = r2.compute_similarity(n=n_retrieve)
    fl = r2.get_result(d2, c2, q2, re_rank=False)
    for i in range(len(fl)):
        fl[i]['fact'] = rerank_facts(q2[i], fl[i].get('fact') or [])

    print('[4/4] Reasoning (Qwen fine-tuned)...')
    results, correct = [], 0
    for i, item in enumerate(tqdm(qs, desc='Generate', leave=False)):
        facts = (fl[i].get('fact') or [])[:top_k_facts]
        gold  = get_gold(qs_raw[i])
        pred  = gen_answer(build_reasoning_prompt(facts, get_q(item)))
        ok = evaluate_hit(pred, gold)
        correct += ok
        results.append({'question':get_q(qs_raw[i]), 'q_norm':q_norm[i], 'gold':str(gold),
                        'predicted':pred, 'parsed':parse_list_answer(pred),
                        'correct':ok, 'retrieved_facts':facts})
    em = correct/len(results)*100
    print(f'\n✅ {label}: {correct}/{len(results)} = {em:.2f}%')
    return results, em

print('✅ Pipeline v8 ready')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — CHẠY: Qwen FINE-TUNED (1000 câu) + Qwen BASE (subset) để ablation
# ═══════════════════════════════════════════════════════════
TEST_N      = 1000    # giảm nếu thiếu giờ
ABLATION_N  = 200     # subset đo base (chưa fine-tune) cho rẻ

# (A) Fine-tuned trên toàn bộ test
USE_FT = True
res_ft, em_ft = run_pipeline_v8(questions_1000[:TEST_N], triple_list, RETRIEVER_PATH,
                                label=f'Qwen2.5 FINE-TUNED (n={TEST_N})')
with open('/kaggle/working/results_v8.json','w',encoding='utf-8') as f:
    json.dump(res_ft, f, ensure_ascii=False, indent=2)

# (B) Base (chưa fine-tune) trên subset → đo delta do fine-tune
USE_FT = False
res_base, em_base = run_pipeline_v8(questions_1000[:ABLATION_N], triple_list, RETRIEVER_PATH,
                                    label=f'Qwen2.5 BASE (n={ABLATION_N}, ablation)')
USE_FT = True

# fine-tuned trên cùng subset để so công bằng
res_ft_sub = res_ft[:ABLATION_N]
em_ft_sub = sum(x['correct'] for x in res_ft_sub)/len(res_ft_sub)*100

print('\n' + '═'*56)
print('  ABLATION (cùng subset, cùng retriever/pipeline)')
print('═'*56)
print(f'  Qwen BASE       : {em_base:.2f}%')
print(f'  Qwen FINE-TUNED : {em_ft_sub:.2f}%')
print(f'  → Tác động fine-tune LLM: {em_ft_sub-em_base:+.2f} điểm')


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — BẢNG SO SÁNH + LƯU SUMMARY
# ═══════════════════════════════════════════════════════════
print('═'*70)
print('  HÀNH TRÌNH — BÁO CÁO VỚI THẦY')
print('═'*70)
rows = [
    ('TimeR4 paper (LLaMA2 fine-tune, đầy đủ)', '72.78%', '— (mốc trần)'),
    ('TimeR4 paper (LLaMA2/ChatGPT KHÔNG fine-tune)', '39–41%', '— (mốc so sánh)'),
    ('v4 FULL FIX (Mistral)', '15.40%', ''),
    ('v5 tăng recall (Mistral)', '16.60%', ''),
    ('v6 đa ngôn ngữ + directionality (Mistral)', '16.20%', ''),
    ('v7 rerank thời gian (Mistral)', '16.70%', ''),
    ('v8 Qwen2.5 FINE-TUNE (notebook này)', f'{em_ft:.2f}%', f'{em_ft-16.70:+.2f} vs v7'),
]
for name, h, note in rows:
    print(f'  {name:<48} {h:>8}  {note}')
print('═'*70)

summary = {
    'em_v7': 16.70,
    'em_v8_qwen_ft': round(em_ft, 2),
    'em_v8_qwen_base_subset': round(em_base, 2),
    'em_v8_qwen_ft_subset': round(em_ft_sub, 2),
    'finetune_gain_subset': round(em_ft_sub - em_base, 2),
    'improvement_over_v7': round(em_ft - 16.70, 2),
    'paper_ceiling_finetune': 72.78,
    'paper_no_finetune': 41.4,
    'test_n': TEST_N,
}
with open('/kaggle/working/summary_v8_FINAL.json','w',encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('✅ summary_v8_FINAL.json + results_v8.json saved → tab Output')
